# Day 3: What goes with each pathway?

**Research question 2:** Which characteristics are associated with each energy pathway, and do on-site generation and purchasing have different profiles?

Day 2 showed the models can tell the pathways apart moderately well. Today we open the model with the best cross-validated score (XGBoost on all predictors) and ask *what it is using* to do so.

The tool is **SHAP** (SHapley Additive exPlanations). For every firm and every predictor, SHAP gives one number: how much that predictor moved the model's score for a pathway up or down for that firm. Positive means the model scored the pathway as more likely, negative as less likely. The score is the model's raw score before it is turned into a probability (not a percentage), and for each firm the numbers, added to the model's starting value, give exactly that score.

**Two limits to keep in mind and to state in the paper.** SHAP explains the model, not the world: it shows which characteristics go together with a pathway, not that changing a characteristic would change a firm's pathway. And when predictors are related to each other, the way the model shares the credit between them is not the only possible one.

**How to run it:** *Runtime > Run all*, then upload `sme_energy_clean.csv`. It takes about one minute.

## 0. Setup, and rebuild the Day 2 model

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)
SEED = 42

DATA_FILE = "sme_energy_clean.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Choose sme_energy_clean.csv")
        DATA_FILE = list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError("Put sme_energy_clean.csv in the same folder as this notebook.")
df = pd.read_csv(DATA_FILE)

CLASS_NAMES = ["No renewable", "Purchased", "On-site"]
y = df["pathway_code"]
df["log_employees"] = np.log1p(df["employees"])
df["log_firm_age"] = np.log1p(df["firm_age_years"].clip(lower=0))

FIRM = ["nace_section", "size", "log_employees", "log_firm_age", "age_band", "turnover_band",
        "turnover_trend", "employment_trend", "hiring_difficulty", "sells"]
BEHAVIOUR = ([c for c in df.columns if c.startswith(("act_", "sup_", "ext_", "bar_", "help_"))]
             + ["n_other_actions", "n_barriers", "asked_followups", "investment", "climate_strategy",
                "green_products", "green_jobs", "recycled_materials", "lifespan_policy"])
PREDICTORS = FIRM + ["country"] + BEHAVIOUR          # predictor set "B All" from Day 2

# Same split as Day 2 (same SEED), so the test firms are the same firms.
train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=y, random_state=SEED)
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print("Predictors:", len(PREDICTORS), "| Training firms:", len(train_idx), "| Test firms:", len(test_idx))

In [ ]:
NUM_WITH_GAPS = ["log_employees", "log_firm_age"]
cat_cols = [c for c in PREDICTORS if not pd.api.types.is_numeric_dtype(df[c])]
num_cols = [c for c in PREDICTORS if pd.api.types.is_numeric_dtype(df[c]) and c not in NUM_WITH_GAPS]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("gaps", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]),
     NUM_WITH_GAPS),
    ("num", StandardScaler(), num_cols)])
X_train = pre.fit_transform(df.loc[train_idx, PREDICTORS])
X_test = pre.transform(df.loc[test_idx, PREDICTORS])
encoded_names = list(pre.get_feature_names_out())

# The settings that won the tuning on Day 2 (see the "Cross-validation" sheet of Day2_results.xlsx).
XGB_SETTINGS = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.6, min_child_weight=1, subsample=0.8)
LOGIT_C = 0.1
model = xgb.XGBClassifier(objective="multi:softprob", tree_method="hist", n_jobs=2, random_state=SEED, verbosity=0, **XGB_SETTINGS)
model.fit(X_train, y_train, sample_weight=compute_sample_weight("balanced", y_train))
auc = roc_auc_score(y_test, model.predict_proba(X_test), multi_class="ovr")
print(f"Test AUC: {auc:.3f}   (this should match the XGBoost score for 'B All' in your Day 2 results)")

## 1. Compute the SHAP values

XGBoost can compute exact SHAP values for its own trees (`pred_contribs=True`). The result has one number per test firm, per pathway, per encoded column.

The check below confirms the defining property: for every firm, the SHAP values plus a baseline add up to the model's score.

In [ ]:
booster = model.get_booster()
dtest = xgb.DMatrix(X_test)
contrib = booster.predict(dtest, pred_contribs=True)            # firms x pathways x (encoded columns + 1 baseline)
shap_encoded, baseline = contrib[:, :, :-1], contrib[:, :, -1]
margin = booster.predict(dtest, output_margin=True)
print("Shape:", shap_encoded.shape, "= (test firms, pathways, encoded columns)")
print("Largest gap between SHAP sum and model score:", float(np.abs(shap_encoded.sum(axis=2) + baseline - margin).max()))

### From encoded columns back to the original predictors

One-hot encoding turned `country` into 20 columns, `investment` into 8, and so on. Because SHAP values add up, we can sum the columns that belong to one predictor and get that predictor's total contribution for each firm.

In [ ]:
def original_predictor(encoded):
    block, rest = encoded.split("__", 1)
    if block == "cat":
        return max((c for c in cat_cols if rest.startswith(c + "_")), key=len)
    if rest.startswith("missingindicator_"):
        return rest.replace("missingindicator_", "")
    return rest

owner = [original_predictor(n) for n in encoded_names]
shap = np.zeros((len(test_idx), 3, len(PREDICTORS)))
for j, p in enumerate(owner):
    shap[:, :, PREDICTORS.index(p)] += shap_encoded[:, :, j]
print("Shape after grouping:", shap.shape, "= (test firms, pathways, predictors)")

## 2. Which predictors matter most?

**Importance** of a predictor for a pathway = the average size of its SHAP values across firms, ignoring the sign. A large value means the predictor often moves the prediction a lot, in either direction.

In [ ]:
importance = pd.DataFrame(np.abs(shap).mean(axis=0).T, index=PREDICTORS, columns=CLASS_NAMES)
importance["Average"] = importance.mean(axis=1)
importance = importance.sort_values("Average", ascending=False).round(4)
importance.head(15)

### Grouped into themes

57 predictors are hard to read at once. Here they are grouped into 13 themes. A theme's importance is the average size of the *summed* SHAP values of its members.

In [ ]:
def theme_of(p):
    if p == "country": return "Country"
    if p in ("size", "log_employees"): return "Firm size"
    if p in ("turnover_band", "turnover_trend", "employment_trend"): return "Turnover and growth"
    if p in ("nace_section", "sells"): return "Sector and offer"
    if p in ("log_firm_age", "age_band"): return "Firm age"
    if p == "hiring_difficulty": return "Hiring difficulty"
    if p.startswith("act_") or p in ("n_other_actions", "asked_followups"): return "Other green actions"
    if p == "investment": return "Investment level"
    if p.startswith(("sup_", "ext_")): return "Support used"
    if p.startswith("bar_") or p == "n_barriers": return "Barriers met"
    if p.startswith("help_"): return "Support wanted"
    if p == "climate_strategy": return "Climate strategy"
    return "Green products, jobs, materials"

themes = sorted(set(theme_of(p) for p in PREDICTORS))
shap_theme = np.zeros((len(test_idx), 3, len(themes)))
for j, p in enumerate(PREDICTORS):
    shap_theme[:, :, themes.index(theme_of(p))] += shap[:, :, j]
theme_importance = pd.DataFrame(np.abs(shap_theme).mean(axis=0).T, index=themes, columns=CLASS_NAMES)
theme_importance["Predictors in theme"] = [sum(theme_of(p) == t for p in PREDICTORS) for t in themes]
theme_importance = theme_importance.sort_values("On-site", ascending=False)
theme_importance.round(4)

## 3. Do the two renewable pathways have different profiles?

This is the centre of the paper. For each theme we compare its importance for **Purchased** with its importance for **On-site**.

- The **ratio** column is on-site importance divided by purchased importance. Above 1 means the model relies on the theme more for generating; below 1, more for buying. A ratio is unstable when the purchased importance is close to zero, so read its direction, not its size. Day 5 checks the direction across 20 different splits.
- The **rank correlation** summarises all 57 predictors in one number: 1 would mean both pathways rank the predictors identically, 0 would mean the rankings are unrelated.

In [ ]:
compare = theme_importance[["Purchased", "On-site"]].copy()
compare["Ratio (on-site / purchased)"] = compare["On-site"] / compare["Purchased"]
compare["Matters more for"] = np.where(compare["Ratio (on-site / purchased)"] > 1.25, "On-site",
                              np.where(compare["Ratio (on-site / purchased)"] < 0.8, "Purchased", "Both similarly"))
rho, _ = spearmanr(importance["Purchased"], importance["On-site"])
print(f"Rank correlation of predictor importance, Purchased vs On-site: {rho:.2f}")
compare.sort_values("Ratio (on-site / purchased)", ascending=False).round(3)

### Figure 5: importance of each theme for the two renewable pathways

In [ ]:
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": GRID, "text.color": INK,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK})

t = theme_importance.assign(m=theme_importance[["Purchased", "On-site"]].max(axis=1)).sort_values("m")
ypos = np.arange(len(t))
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.barh(ypos + 0.19, t["Purchased"], height=0.36, color=BLUE, label="Purchased from a supplier")
ax.barh(ypos - 0.19, t["On-site"], height=0.36, color=ORANGE, label="Generated on-site")
ax.set_yticks(ypos); ax.set_yticklabels(t.index)
ax.set_xlabel("Importance (average absolute SHAP value, test set)")
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.tick_params(length=0)
ax.legend(frameon=False, loc="lower right")
ax.set_title("What the model uses to identify each pathway", loc="left", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig("fig5_theme_importance.png", dpi=300); plt.show()

## 4. In which direction does each predictor push?

Importance says *how much* the model relies on a predictor. Direction says *which way*. For a predictor with categories, we average the SHAP values of the firms in each category. Positive means the model scores firms in that category higher for the pathway.

In [ ]:
test = df.loc[test_idx].reset_index(drop=True)

def direction(predictor, nd=3):
    # Average SHAP value by category (or by low/high half for a number), for each pathway.
    j = PREDICTORS.index(predictor)
    values = test[predictor]
    if pd.api.types.is_numeric_dtype(values) and values.nunique() > 2:
        groups = np.where(values.isna(), "missing", np.where(values <= values.median(), "low half", "high half"))
    else:
        groups = values.astype(str).values
    out = pd.DataFrame(shap[:, :, j], columns=CLASS_NAMES).groupby(groups).mean().round(nd)
    out.insert(0, "Firms", pd.Series(groups).value_counts())
    return out

for p in ["size", "log_employees", "log_firm_age", "investment", "climate_strategy"]:
    print(p); display(direction(p))

In [ ]:
# Yes/no predictors: the difference in average SHAP between firms answering yes and firms answering no.
rows = []
for p in PREDICTORS:
    v = test[p]
    if pd.api.types.is_numeric_dtype(v) and set(v.dropna().unique()) <= {0, 1}:
        j = PREDICTORS.index(p)
        d = shap[v.values == 1][:, :, j].mean(axis=0) - shap[v.values == 0][:, :, j].mean(axis=0)
        rows.append({"Predictor": p, "Firms answering yes": int(v.sum()),
                     **{f"Effect of yes: {n}": float(d[c]) for c, n in enumerate(CLASS_NAMES)}})
yes_no = pd.DataFrame(rows).set_index("Predictor")
yes_no["Strongest effect"] = yes_no[[c for c in yes_no.columns if c.startswith("Effect")]].abs().max(axis=1)
yes_no = yes_no.sort_values("Strongest effect", ascending=False).drop(columns="Strongest effect")
yes_no.head(12).round(3)

### Figure 6: the contribution of country to each pathway

Each bar is the average amount the model adds to a firm's score for being in that country, after accounting for everything else the model knows about the firm. The unit is the model's raw score (the value that the model turns into a probability), not percentage points. A bar to the right means a higher score for the pathway; to the left, a lower one. Both panels list the countries in the same order.

In [ ]:
country_effect = direction("country")
c = country_effect.sort_values("Purchased")
fig, axes = plt.subplots(1, 2, figsize=(7.4, 6.6), sharey=True)
lim = np.abs(c[["Purchased", "On-site"]].values).max() * 1.12
fig.set_size_inches(7.4, 1.2 + 0.24 * len(c))
for ax, col, colour, title in [(axes[0], "Purchased", BLUE, "Purchasing"),
                               (axes[1], "On-site", ORANGE, "On-site generation")]:
    ax.barh(c.index, c[col], color=colour, height=0.62)
    ax.axvline(0, color=MUTED, linewidth=1)
    ax.set_xlim(-lim, lim)
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    ax.set_xlabel("Average SHAP value (raw model score)")
    ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
    ax.tick_params(length=0)
fig.suptitle("Contribution of country to the model score (countries ordered by purchasing)", x=0.02, ha="left",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig("fig6_country_effects.png", dpi=300); plt.show()

# A check that does not depend on the model: the raw share of firms on each pathway, by country (all firms).
share = pd.crosstab(df["country"], df["pathway"], normalize="index")
r_share, p_share = spearmanr(share["Purchased"], share["On-site"])
print(f"Across the {len(share)} countries, rank correlation between the unweighted share purchasing and the share generating: "
      f"{r_share:.2f} (p = {p_share:.2f})")
check = country_effect.join(share, rsuffix=" share")
rho_share_p = spearmanr(check['Purchased'], check['Purchased share'])[0]; rho_share_o = spearmanr(check['On-site'], check['On-site share'])[0]
print("Does the model's country contribution follow the unweighted country share?  Purchased: rho = "
      f"{rho_share_p:.2f}   On-site: rho = {rho_share_o:.2f}")
print("(It should: country is one of the predictors. This shows the model has not distorted the pattern; it is not independent evidence.)")

## 5. Cross-check against logistic regression

If the findings do not depend on the kind of model, a completely different model should give a similar picture. Logistic regression is a linear model, and for a linear model the SHAP value has a simple formula: coefficient × (the firm's value − the average value), also on the scale of the raw score. The two models' scores are not calibrated to each other, so we do not compare their importance values directly. We compare the *rankings* and the *direction of the ratio*.

In [ ]:
logit = LogisticRegression(max_iter=3000, class_weight="balanced", C=LOGIT_C).fit(X_train, y_train)
centred = X_test - X_train.mean(axis=0)
lin_encoded = centred[:, None, :] * logit.coef_[None, :, :]                 # firms x pathways x encoded columns
lin = np.zeros_like(shap)
for j, p in enumerate(owner):
    lin[:, :, PREDICTORS.index(p)] += lin_encoded[:, :, j]
lin_theme = np.zeros_like(shap_theme)
for j, p in enumerate(PREDICTORS):
    lin_theme[:, :, themes.index(theme_of(p))] += lin[:, :, j]

logit_importance = pd.DataFrame(np.abs(lin).mean(axis=0).T, index=PREDICTORS, columns=CLASS_NAMES)
logit_theme = pd.DataFrame(np.abs(lin_theme).mean(axis=0).T, index=themes, columns=CLASS_NAMES)

rows = []
for n in CLASS_NAMES:
    r_pred, _ = spearmanr(importance.loc[PREDICTORS, n], logit_importance.loc[PREDICTORS, n])
    r_theme, _ = spearmanr(theme_importance.loc[themes, n], logit_theme.loc[themes, n])
    top_x = set(importance[n].sort_values(ascending=False).index[:10])
    top_l = set(logit_importance[n].sort_values(ascending=False).index[:10])
    rows.append({"Pathway": n, "Rank correlation (57 predictors)": r_pred,
                 "Rank correlation (13 themes)": r_theme, "Shared predictors in top 10": len(top_x & top_l)})
agreement = pd.DataFrame(rows).set_index("Pathway")

logit_compare = logit_theme[["Purchased", "On-site"]].copy()
logit_compare["Ratio (on-site / purchased)"] = logit_compare["On-site"] / logit_compare["Purchased"]
logit_compare = logit_compare.loc[compare.sort_values("Ratio (on-site / purchased)", ascending=False).index]
same_side = (logit_compare["Ratio (on-site / purchased)"] > 1) == (compare.loc[logit_compare.index, "Ratio (on-site / purchased)"] > 1)
display(agreement.round(2))
print("Themes where the two models put the ratio on the same side of 1: %d of %d   (they differ for: %s)" % (
    same_side.sum(), len(same_side), ", ".join(same_side.index[~same_side]) or "none"))
print("Logistic regression: theme importance for the two renewable pathways (same order as the XGBoost table)")
logit_compare.round(3)

## 6. YOUR TURN

Pick any predictor and see which way each of its categories pushes. Names to try: `"turnover_band"`, `"nace_section"`, `"green_products"`, `"hiring_difficulty"`, `"age_band"`, `"recycled_materials"`, `"log_employees"`.

For each one, write a sentence of the form: *"Firms that ___ are pushed towards ___ and away from ___."*

In [ ]:
LOOK_AT = "turnover_band"        # <-- change this and re-run

d = direction(LOOK_AT)
display(d)
ax = d[["Purchased", "On-site"]].plot.barh(color=[BLUE, ORANGE], figsize=(6.5, 0.45 * len(d) + 1.4), width=0.75)
ax.axvline(0, color=MUTED, linewidth=1); ax.set_xlabel("Average SHAP value"); ax.set_ylabel("")
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.tick_params(length=0); ax.legend(frameon=False); ax.invert_yaxis()
ax.set_title(LOOK_AT, loc="left", fontweight="bold"); plt.tight_layout(); plt.show()

## 7. Save the outputs

- `Day3_results.xlsx`: every table above, one per sheet.
- `country_effects.csv`: the country effect on each pathway. Day 4 uses it to ask *what about a country* matters.
- `fig5_theme_importance.png`, `fig6_country_effects.png`.

In [ ]:
country_effect.rename_axis("country").to_csv("country_effects.csv")
direction_tables = {p: direction(p, nd=6) for p in cat_cols + ["log_employees", "log_firm_age"]}
table4 = pd.DataFrame({"Purchased": compare["Purchased"], "On-site": compare["On-site"], "Ratio, XGBoost": compare["Ratio (on-site / purchased)"],
                       "Ratio, logistic": logit_compare["Ratio (on-site / purchased)"], "Logistic: purchased": logit_compare["Purchased"],
                       "Logistic: on-site": logit_compare["On-site"], "Predictors in theme": theme_importance["Predictors in theme"]}).sort_values("On-site", ascending=False)
key = pd.DataFrame({"Item": ["Test AUC of the interpreted model", "Largest gap between SHAP sum and model score", "Rank correlation of predictor importance, purchased vs on-site",
                             "Country contribution vs unweighted share, purchased (rho)", "Country contribution vs unweighted share, on-site (rho)",
                             "Themes with the ratio on the same side of 1 in both models", "Themes where the models differ", "Test firms", "Encoded columns"],
                    "Value": [round(auc, 6), float(np.abs(shap_encoded.sum(axis=2) + baseline - margin).max()), round(rho, 6), round(rho_share_p, 6), round(rho_share_o, 6),
                              int(same_side.sum()), ", ".join(same_side.index[~same_side]) or "none", len(test_idx), X_test.shape[1]]})

with pd.ExcelWriter("Day3_results.xlsx") as xl:
    key.to_excel(xl, sheet_name="Key numbers", index=False)
    table4.round(6).to_excel(xl, sheet_name="Table 4")
    importance.to_excel(xl, sheet_name="Predictor importance")
    theme_importance.round(4).to_excel(xl, sheet_name="Theme importance")
    compare.sort_values("Ratio (on-site / purchased)", ascending=False).round(4).to_excel(xl, sheet_name="Purchased vs on-site")
    yes_no.round(6).to_excel(xl, sheet_name="Yes-no effects")
    direction("country", nd=6).to_excel(xl, sheet_name="Country effects")
    agreement.to_excel(xl, sheet_name="Agreement with logit")
    logit_compare.round(4).to_excel(xl, sheet_name="Logit theme importance")
    row = 0
    for p, tab in direction_tables.items():
        if p == "country": continue
        pd.DataFrame({"Predictor": [p]}).to_excel(xl, sheet_name="Direction by category", startrow=row, index=False)
        tab.to_excel(xl, sheet_name="Direction by category", startrow=row + 2)
        row += len(tab) + 5
print("Saved.")

try:
    from google.colab import files
    for f in ["Day3_results.xlsx", "country_effects.csv", "fig5_theme_importance.png", "fig6_country_effects.png"]:
        files.download(f)
except ImportError:
    pass